# Paper figures

Builds every figure and table used in the paper from **saved** experiment outputs. No simulation runs here: rerun `run_mpc.ipynb` / `run_irf.ipynb` first, then rerun this notebook.

All files are written to `data/output_data/paper-figures/`.

| Figure | Source run | Status |
|---|---|---|
| Impact MPC distributions (12 panels) | `run_mpc.ipynb`, `write_paper_figures` | in paper |
| Aggregate MPC table | `run_mpc.ipynb`, `paper_mpc_summary_filtered.csv` | in paper |
| IRF: monetary policy +100bp | `run_irf.ipynb` | in paper |
| IRF: unemployment +2pp | `run_irf.ipynb` | candidate (C response under review) |
| IRF: one-off transfer, total uncertainty | `run_irf.ipynb` | add to `PAPER_IRF_SHOCKS` once checked |

Excluded until diagnosed: income-tax IRF (wrong sign), individual/common uncertainty IRFs.

In [ ]:
%load_ext autoreload
%autoreload 2

import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

from src.mpc_paper_figures import write_paper_figures
from src.visual_helpers import _prepare_irf_plot_panel

MPC_DIR = Path("data/output_data/mpc-experiment/analysis")
IRF_DIR = Path("data/output_data/irf-experiment/analysis")
OUT = Path("data/output_data/paper-figures")
OUT.mkdir(parents=True, exist_ok=True)
SHOW_TRANSFER_WEIGHTED_MEAN = True

## 1. Impact MPC distributions
Filtered household-seed panel; boxes = distribution, black dots = household means, orange diamonds = transfer-weighted means.

In [ ]:
# Same panel run_mpc.ipynb passes to write_paper_figures (filtered when apply_mpc_filters=True).
panel_path = MPC_DIR / "household_mpc_panel_filtered.csv"
if not panel_path.exists():
    panel_path = MPC_DIR / "household_mpc_panel.csv"
mpc_panel = pd.read_csv(panel_path)
write_paper_figures(mpc_panel, OUT, show_transfer_weighted_mean=SHOW_TRANSFER_WEIGHTED_MEAN)
display(Image(filename=str(OUT / "paper_impact_mpc.png")))
pd.read_csv(OUT / "paper_figure_group_means.csv").round(3)

## 2. Aggregate MPC table
Transfer-weighted MPC = ΣΔC / Σtransfer, real goods consumption; SE clustered by seed. Cumulative MPCs use only households whose labour-income gap between runs is ≤ 5% of the transfer (state the selection in the text).

In [ ]:
summary = pd.read_csv(MPC_DIR / "paper_mpc_summary_filtered.csv")
rows = {
    "real_mpc_impact": "Impact (quarter of transfer)",
    "real_cmpc_2p": "Cumulative, 2 quarters",
    "real_cmpc_3p": "Cumulative, 3 quarters",
    "real_cmpc_4q": "Cumulative, 4 quarters",
}
mpc_table = (
    summary[(summary.variable == "all") & summary.outcome.isin(rows)]
    .set_index("outcome")
    .loc[list(rows), ["transfer_weighted_mpc", "seed_cluster_se", "median", "negative_share", "n", "seeds"]]
    .rename(index=rows)
)
mpc_table.to_csv(OUT / "paper_mpc_table.csv")
(mpc_table.style.format({"transfer_weighted_mpc": "{:.3f}", "seed_cluster_se": "{:.4f}", "median": "{:.3f}", "negative_share": "{:.1%}"}))

In [ ]:
# LaTeX version of the table for the paper.
latex = mpc_table[["transfer_weighted_mpc", "seed_cluster_se", "negative_share", "n"]].to_latex(
    float_format="%.3f", header=["MPC", "SE (seed)", "Share $<0$", "$N$"], escape=False
)
(OUT / "paper_mpc_table.tex").write_text(latex)
print(latex)

## 3. Impulse responses
Mean across seeds (line) and 10th–90th percentile across seeds (band). Deviations from the paired no-shock run: % for real GDP, real consumption and CPI; percentage points for unemployment and the policy rate. Horizon 0 = first quarter with the shock.

In [ ]:
PAPER_IRF_SHOCKS = {
    "policy_rate_100bp": "Policy rate +100bp (4 quarters)",
    "unemp_2pp": "Unemployment +2pp (4 quarters)",
    # Add after review: "household_transfer_10pct": "One-off transfer", "total_income_uncertainty_25pct": "Total income uncertainty +25%",
}
PAPER_IRF_VARIABLES = {
    "real_gdp": "Real GDP, %",
    "real_cons": "Real consumption, %",
    "cpi": "CPI, %",
    "unemp": "Unemployment, pp",
    "policy_rate": "Policy rate, pp",
}
MAX_HORIZON = 16

irf_raw = pd.read_csv(IRF_DIR / "irf_panel.csv")
irf_raw = irf_raw[irf_raw.shock_name.isin(PAPER_IRF_SHOCKS)]
irf = _prepare_irf_plot_panel(irf_raw)
irf = irf[irf.variable.isin(PAPER_IRF_VARIABLES) & (irf.horizon <= MAX_HORIZON)]
bands = (
    irf.groupby(["shock_name", "variable", "horizon"])["plot_value"]
    .agg(mean="mean", p10=lambda x: x.quantile(0.1), p90=lambda x: x.quantile(0.9), n="count")
    .reset_index()
)
bands.to_csv(OUT / "paper_irf_bands.csv", index=False)

def plot_paper_irfs(bands, shocks, variables, path):
    fig, axes = plt.subplots(len(shocks), len(variables), figsize=(3.0 * len(variables), 2.4 * len(shocks)), squeeze=False)
    for i, (shock, shock_label) in enumerate(shocks.items()):
        for j, (var, var_label) in enumerate(variables.items()):
            ax = axes[i, j]
            d = bands[(bands.shock_name == shock) & (bands.variable == var)]
            ax.fill_between(d.horizon, d.p10, d.p90, color="#7a9cbf", alpha=0.35, linewidth=0)
            ax.plot(d.horizon, d["mean"], color="#1f3b5c", linewidth=1.8)
            ax.axhline(0, color="grey", linewidth=0.7)
            ax.spines[["top", "right"]].set_visible(False)
            if i == 0:
                ax.set_title(var_label, fontsize=10)
            if j == 0:
                ax.set_ylabel(shock_label, fontsize=9)
            if i == len(shocks) - 1:
                ax.set_xlabel("Quarters after shock")
    fig.tight_layout()
    fig.savefig(path.with_suffix(".pdf"))
    fig.savefig(path.with_suffix(".png"), dpi=200)
    plt.show()

plot_paper_irfs(bands, PAPER_IRF_SHOCKS, PAPER_IRF_VARIABLES, OUT / "paper_irf_macro")
print("seeds per cell:", bands.n.unique())

In [ ]:
# Peak responses for the text.
peaks = bands.loc[bands.groupby(["shock_name", "variable"])["mean"].apply(lambda s: s.abs().idxmax())]
peaks[["shock_name", "variable", "horizon", "mean", "p10", "p90"]].round(3)